# 06 Export finetuned MapLane CLRKDNet to ONNX - Local

05 노트북에서 lane 후보를 주행 중심선으로 바꾸는 후처리 설정을 1차 확정했다.  
이번 노트북의 목적은 fine-tuned CLRKDNet checkpoint를 로컬에서 ONNX로 변환하고, PyTorch raw output과 ONNX Runtime raw output이 잘 맞는지 확인하는 것이다.

중요한 점:

- 이 ONNX는 **raw forward 모델**만 담는다.
- 즉, ONNX 출력은 steering 값이 아니라 CLRKDNet head의 raw prediction tensor다.
- 공식 `get_lanes()`에 해당하는 lane 후보 decode, 그리고 05에서 정한 driving postprocess는 Pi Python 코드 쪽에서 수행한다.
- 따라서 이번 노트북의 검증 목표는 `PyTorch raw output == ONNX Runtime raw output`에 가깝게 맞는지 확인하는 것이다.

## 0. 실행 환경

이 노트북은 로컬 VSCode에서 `<env>` kernel로 실행한다.

필요한 입력은 이미 로컬에 내려받은 fine-tuning 결과다.

```text
10_experiments/06_lane_model_finetuning/
  colab_outputs/finetune_v1/MapLane_Field1Field2_full/{run}/
    config.py
    ckpt/best.pth
    code/
      clrkd/
      configs/
      mmcv/
```

이전 02~05 노트북에서 fine-tuned checkpoint를 로컬에서 이미 불러왔으므로, 같은 구조를 사용한다.

In [1]:
from pathlib import Path
import os
import sys
import json
import time
import shutil
import types

import numpy as np

PROJECT_ROOT = Path(r'~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization')
EXP_ROOT = PROJECT_ROOT / '10_experiments' / '06_lane_model_finetuning'
OUTPUT_ROOT = EXP_ROOT / 'colab_outputs' / 'finetune_v1'
FULL_WORK_ROOT = OUTPUT_ROOT / 'MapLane_Field1Field2_full'

REVIEW05_TABLE_DIR = EXP_ROOT / 'review_outputs' / '05_driving_postprocess_sweep' / 'tables'
POSTPROCESS_SOURCE_JSON = REVIEW05_TABLE_DIR / 'final_driving_postprocess_candidates.json'

EXPORT_DIR = EXP_ROOT / 'model_exports' / 'finetune_v1'
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

ONNX_PATH = EXPORT_DIR / 'maplane_finetune_v1_fp32_raw.onnx'
EXPORT_META_PATH = EXPORT_DIR / 'maplane_finetune_v1_export_meta.json'
POSTPROCESS_CONFIG_PATH = EXPORT_DIR / 'maplane_finetune_v1_driving_postprocess_config.json'

print('PROJECT_ROOT:', PROJECT_ROOT)
print('FULL_WORK_ROOT:', FULL_WORK_ROOT, 'exists=', FULL_WORK_ROOT.exists())
print('POSTPROCESS_SOURCE_JSON:', POSTPROCESS_SOURCE_JSON, 'exists=', POSTPROCESS_SOURCE_JSON.exists())
print('EXPORT_DIR:', EXPORT_DIR)

assert FULL_WORK_ROOT.exists(), FULL_WORK_ROOT
assert POSTPROCESS_SOURCE_JSON.exists(), POSTPROCESS_SOURCE_JSON

PROJECT_ROOT: ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization
FULL_WORK_ROOT: ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\06_lane_model_finetuning\colab_outputs\finetune_v1\MapLane_Field1Field2_full exists= True
POSTPROCESS_SOURCE_JSON: ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\06_lane_model_finetuning\review_outputs\05_driving_postprocess_sweep\tables\final_driving_postprocess_candidates.json exists= True
EXPORT_DIR: ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\06_lane_model_finetuning\model_exports\finetune_v1


In [2]:
run_dirs = [
    p for p in FULL_WORK_ROOT.iterdir()
    if p.is_dir() and (p / 'ckpt' / 'best.pth').exists() and (p / 'config.py').exists() and (p / 'code').exists()
]
run_dirs = sorted(run_dirs, key=lambda p: p.stat().st_mtime)

print('candidate full runs:')
for p in run_dirs:
    print('-', p.name)

assert run_dirs, 'No full run directory containing code/config.py/ckpt/best.pth was found.'
RUN_DIR = run_dirs[-1]
CHECKPOINT_PATH = RUN_DIR / 'ckpt' / 'best.pth'
CONFIG_PATH = RUN_DIR / 'config.py'
CODE_DIR = RUN_DIR / 'code'

print('\nRUN_DIR:', RUN_DIR)
print('CHECKPOINT_PATH:', CHECKPOINT_PATH, 'size_MB=', round(CHECKPOINT_PATH.stat().st_size / 1024 / 1024, 2))
print('CONFIG_PATH:', CONFIG_PATH)
print('CODE_DIR:', CODE_DIR)

candidate full runs:
- 20260505_201647_lr_1e-04_b_8

RUN_DIR: ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\06_lane_model_finetuning\colab_outputs\finetune_v1\MapLane_Field1Field2_full\20260505_201647_lr_1e-04_b_8
CHECKPOINT_PATH: ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\06_lane_model_finetuning\colab_outputs\finetune_v1\MapLane_Field1Field2_full\20260505_201647_lr_1e-04_b_8\ckpt\best.pth size_MB= 131.63
CONFIG_PATH: ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\06_lane_model_finetuning\colab_outputs\finetune_v1\MapLane_Field1Field2_full\20260505_201647_lr_1e-04_b_8\config.py
CODE_DIR: ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\06_lane_model_finetuning\colab_outputs\finetune_v1\MapLane_Field1Field2_full\20260505_201647_lr_1e-04_b_8\code


## STOP-CHECK A

위 셀에서 `RUN_DIR`, `CHECKPOINT_PATH`, `CONFIG_PATH`, `CODE_DIR`가 올바르게 잡혔는지 확인한다.  
`best.pth`가 아니라 특정 epoch를 export하고 싶으면 `CHECKPOINT_PATH`만 수동으로 바꾸면 된다.

## 1. Local dependency 확인

ONNX export에는 `torch`, `onnx`, `onnxruntime`, `onnxscript`가 필요하다.  
없는 패키지가 나오면 `<env>`에 설치한 뒤 다시 실행하면 된다.

## 1-1. 필요한 패키지 설치

`onnxscript`가 없으면 PyTorch의 ONNX export가 실패할 수 있다.  
아래 셀은 현재 kernel인 `<env>`에 필요한 export dependency를 설치한다.

이미 설치되어 있으면 빠르게 지나간다.

In [3]:
# %pip install -q onnxscript
# %pip install -q timm

In [4]:
import importlib

mods = ['torch', 'onnx', 'onnxruntime', 'onnxscript', 'cv2', 'numpy', 'timm']
for name in mods:
    try:
        mod = importlib.import_module(name)
        print(name, 'OK', getattr(mod, '__version__', ''))
    except Exception as e:
        print(name, 'FAIL', type(e).__name__, str(e)[:300])
        raise

torch OK 2.11.0+cpu
onnx OK 1.21.0
onnxruntime OK 1.23.2
onnxscript OK 0.7.0
cv2 OK 4.13.0
numpy OK 2.2.6
timm OK 1.0.26


## 2. CLRKDNet code snapshot 로딩

학습 결과 폴더 안의 `code/` snapshot을 사용한다.  
이렇게 해야 fine-tuning 당시 사용한 shim/config/code와 최대한 같은 상태에서 export할 수 있다.

In [5]:
import torch


def win_long_path(path):
    path = Path(path)
    text = str(path)
    if os.name == 'nt' and not text.startswith('\\\\?\\'):
        return '\\\\?\\' + text
    return text


os.chdir(str(CODE_DIR))
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

# Raw forward export does not call NMS, but CLRKDNet imports the custom extension.
fake_nms_impl = types.ModuleType('clrkd.ops.nms_impl')
def _nms_forward(*args, **kwargs):
    raise RuntimeError('NMS extension is unavailable. Raw ONNX export does not call NMS.')
fake_nms_impl.nms_forward = _nms_forward
sys.modules['clrkd.ops.nms_impl'] = fake_nms_impl


from clrkd.utils.config import Config

# Windows-safe patch for CLRKDNet/MMCV Config.fromfile().
# The original implementation creates NamedTemporaryFile and then copies over
# the still-open file. That works on Linux/Colab but fails on Windows.
import importlib.util
import shutil
import tempfile
import os.path as osp


def _file2dict_windows_safe(filename):
    filename = osp.abspath(osp.expanduser(str(filename)))
    if not osp.isfile(filename):
        raise FileNotFoundError(f'config file does not exist: {filename}')

    if filename.endswith('.py'):
        Config._validate_py_syntax(filename)
        with tempfile.TemporaryDirectory() as temp_config_dir:
            temp_config_path = Path(temp_config_dir) / 'tmp_config.py'
            shutil.copyfile(filename, temp_config_path)
            temp_module_name = f'_tmp_clrkd_config_{abs(hash(filename))}'
            spec = importlib.util.spec_from_file_location(temp_module_name, str(temp_config_path))
            mod = importlib.util.module_from_spec(spec)
            sys.modules[temp_module_name] = mod
            assert spec.loader is not None
            spec.loader.exec_module(mod)
            cfg_dict = {
                name: value
                for name, value in mod.__dict__.items()
                if not name.startswith('__')
            }
            sys.modules.pop(temp_module_name, None)
    elif filename.endswith(('.yml', '.yaml', '.json')):
        import mmcv
        cfg_dict = mmcv.load(filename)
    else:
        raise IOError('Only py/yml/yaml/json type are supported now!')

    cfg_text = Path(filename).read_text(encoding='utf-8')
    return cfg_dict, cfg_text


Config._file2dict = staticmethod(_file2dict_windows_safe)

import clrkd.models
from clrkd.models.registry import build_net

print('cwd:', Path.cwd())
print('code snapshot loaded')

cwd: ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\06_lane_model_finetuning\colab_outputs\finetune_v1\MapLane_Field1Field2_full\20260505_201647_lr_1e-04_b_8\code
code snapshot loaded


## 3. Fine-tuned checkpoint 로딩

`config.py`는 fine-tuning 당시의 map geometry를 담고 있다.

```text
raw frame: 1296 x 972
cut_height: 445
model input: 800 x 320
```

ONNX 입력도 이 설정을 따른다.

In [6]:
def clean_state_dict(raw):
    state = raw.get('net', raw) if isinstance(raw, dict) else raw
    cleaned = {}
    for key, value in state.items():
        cleaned[key[7:] if key.startswith('module.') else key] = value
    return cleaned


cfg = Config.fromfile(win_long_path(CONFIG_PATH))
cfg.backbone.pretrained = False

model = build_net(cfg)
checkpoint = torch.load(win_long_path(CHECKPOINT_PATH), map_location='cpu')
state = clean_state_dict(checkpoint)
load_result = model.load_state_dict(state, strict=False)
model.eval()

print('cfg img:', cfg.img_w, cfg.img_h)
print('cfg ori:', cfg.ori_img_w, cfg.ori_img_h)
print('cfg cut_height:', cfg.cut_height)
print('missing keys:', len(load_result.missing_keys), load_result.missing_keys[:5])
print('unexpected keys:', len(load_result.unexpected_keys), load_result.unexpected_keys[:5])

cfg img: 800 320
cfg ori: 1296 972
cfg cut_height: 445
missing keys: 0 []
unexpected keys: 0 []


## STOP-CHECK B

`missing keys`와 `unexpected keys`가 0에 가까워야 한다.  
여기서 문제가 크면 export를 진행하지 말고 checkpoint/config 경로를 다시 확인한다.

## 4. Raw forward wrapper 확인

CLRKDNet의 일반 inference는 raw output 이후 공식 `get_lanes()`를 호출한다.  
하지만 ONNX로 내보낼 때는 raw output까지만 포함한다.

Pi에서는 아래 순서가 된다.

```text
camera frame
-> crop/resize/preprocess
-> ONNX raw_predictions
-> lane decode / get_lanes 대응 후처리
-> 05에서 정한 driving postprocess
-> motor command
```

In [7]:
class RawForwardWrapper(torch.nn.Module):
    def __init__(self, net):
        super().__init__()
        self.net = net

    def forward(self, image):
        output = self.net(image)
        if isinstance(output, (list, tuple)):
            output = output[-1]
        return output


wrapper = RawForwardWrapper(model).eval()
dummy = torch.randn(1, 3, int(cfg.img_h), int(cfg.img_w), dtype=torch.float32)

with torch.no_grad():
    pytorch_output = wrapper(dummy)

print('PyTorch output shape:', tuple(pytorch_output.shape))
print('PyTorch output dtype:', pytorch_output.dtype)
print('PyTorch output min/max:', float(pytorch_output.min()), float(pytorch_output.max()))

PyTorch output shape: (1, 192, 78)
PyTorch output dtype: torch.float32
PyTorch output min/max: -3.6631693840026855 3.658529043197632


## 5. ONNX export

입력 shape는 고정 `1 x 3 x 320 x 800`으로 둔다.  
최종 Pi 주행은 batch=1만 사용하므로 dynamic batch가 꼭 필요하지 않다.

In [8]:
import onnx

t0 = time.time()
torch.onnx.export(
    wrapper,
    dummy,
    win_long_path(ONNX_PATH),
    input_names=['image'],
    output_names=['raw_predictions'],
    opset_version=16,
    do_constant_folding=True,
    dynamo=False,
)
elapsed = time.time() - t0
size_mb = ONNX_PATH.stat().st_size / (1024 * 1024)

print('Exported:', ONNX_PATH)
print(f'ONNX size: {size_mb:.2f} MB')
print(f'export elapsed: {elapsed:.1f} sec')

~\AppData\Local\Temp\ipykernel_8072\422211453.py:4: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


Exported: ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\06_lane_model_finetuning\model_exports\finetune_v1\maplane_finetune_v1_fp32_raw.onnx
ONNX size: 43.95 MB
export elapsed: 1.0 sec


## 6. ONNX checker + PyTorch/ONNX parity

여기서 보는 것은 주행 성능이 아니라 변환 정확도다.  
같은 dummy input에 대해 PyTorch raw output과 ONNX Runtime raw output의 차이가 작아야 한다.

In [9]:
import onnxruntime as ort

onnx_model = onnx.load(win_long_path(ONNX_PATH))
onnx.checker.check_model(onnx_model)
print('onnx checker: OK')

session = ort.InferenceSession(win_long_path(ONNX_PATH), providers=['CPUExecutionProvider'])
input_name = session.get_inputs()[0].name
output_name = session.get_outputs()[0].name

ort_output = session.run([output_name], {input_name: dummy.numpy()})[0]
pt_output = pytorch_output.detach().cpu().numpy()
abs_diff = np.abs(pt_output - ort_output)

print('ONNX input:', input_name, session.get_inputs()[0].shape, session.get_inputs()[0].type)
print('ONNX output:', output_name, session.get_outputs()[0].shape, session.get_outputs()[0].type)
print('max_abs_diff:', float(abs_diff.max()))
print('mean_abs_diff:', float(abs_diff.mean()))
print('p95_abs_diff:', float(np.quantile(abs_diff, 0.95)))

onnx checker: OK
ONNX input: image [1, 3, 320, 800] tensor(float)
ONNX output: raw_predictions [1, 192, 78] tensor(float)
max_abs_diff: 1.9073486328125e-06
mean_abs_diff: 6.825537468557741e-08
p95_abs_diff: 1.9371509552001953e-07


## 7. 선택 이미지 smoke test

원하면 `SAMPLE_IMAGE_PATH`에 이미지 경로를 넣어 실제 이미지 한 장도 통과시켜볼 수 있다.  
이 셀은 선택 사항이다.

주의: 이 테스트는 lane이 잘 그려지는지 판단하는 단계가 아니다.  
이미지가 모델 입력 크기로 들어가고 ONNX가 raw output을 내는지만 확인한다.

In [10]:
import cv2

SAMPLE_IMAGE_PATH = None  # 예: Path(r'C:\path\to\sample.jpg')


def preprocess_bgr_for_model(image_bgr, cfg):
    # Training/inference pipeline uses BGR image -> crop -> resize -> CHW float32 tensor.
    crop_top = int(cfg.cut_height)
    cropped = image_bgr[crop_top:, :, :]
    resized = cv2.resize(cropped, (int(cfg.img_w), int(cfg.img_h)), interpolation=cv2.INTER_LINEAR)
    tensor = resized.astype(np.float32).transpose(2, 0, 1)[None, ...]
    return tensor, resized, crop_top


if SAMPLE_IMAGE_PATH is None:
    print('SAMPLE_IMAGE_PATH is None. Skipping image smoke test.')
else:
    image = cv2.imread(str(SAMPLE_IMAGE_PATH), cv2.IMREAD_COLOR)
    assert image is not None, f'failed to read image: {SAMPLE_IMAGE_PATH}'
    tensor, resized, crop_top = preprocess_bgr_for_model(image, cfg)
    raw = session.run([output_name], {input_name: tensor})[0]
    print('sample image:', SAMPLE_IMAGE_PATH)
    print('crop_top:', crop_top)
    print('input tensor:', tensor.shape, tensor.dtype, float(tensor.min()), float(tensor.max()))
    print('raw output:', raw.shape, raw.dtype, float(raw.min()), float(raw.max()))

SAMPLE_IMAGE_PATH is None. Skipping image smoke test.


## 8. Export metadata와 driving postprocess config 저장

05 노트북에서 고른 후처리 설정을 함께 JSON으로 저장한다.  
Pi 코드에서는 ONNX 모델과 이 설정을 함께 가져가면 된다.

In [11]:
postprocess_source = json.loads(POSTPROCESS_SOURCE_JSON.read_text(encoding='utf-8'))
DRIVING_POSTPROCESS_CONFIG = postprocess_source['best_single_frame']
DRIVING_POSTPROCESS_CONFIG['runtime_policy_note'] = (
    'both lane normal speed; left_only/right_only should use lower confidence and reduced speed; '
    'lost should slow/stop after short hold.'
)

EXPORT_META = {
    'created_at': time.strftime('%Y-%m-%d %H:%M:%S'),
    'checkpoint_path': str(CHECKPOINT_PATH),
    'config_path': str(CONFIG_PATH),
    'code_dir': str(CODE_DIR),
    'onnx_path': str(ONNX_PATH),
    'onnx_size_mb': round(ONNX_PATH.stat().st_size / 1024 / 1024, 3),
    'input': {
        'name': 'image',
        'shape': [1, 3, int(cfg.img_h), int(cfg.img_w)],
        'dtype': 'float32',
        'color_order': 'BGR',
        'normalization': 'none',
    },
    'raw_camera_geometry': {
        'raw_width': int(cfg.ori_img_w),
        'raw_height': int(cfg.ori_img_h),
        'cut_height': int(cfg.cut_height),
        'model_width': int(cfg.img_w),
        'model_height': int(cfg.img_h),
    },
    'output': {
        'name': 'raw_predictions',
        'shape': list(session.get_outputs()[0].shape),
        'meaning': 'CLRKDNet raw head prediction. Lane decode and driving postprocess are performed in Python runtime.',
    },
    'parity': {
        'max_abs_diff': float(abs_diff.max()),
        'mean_abs_diff': float(abs_diff.mean()),
        'p95_abs_diff': float(np.quantile(abs_diff, 0.95)),
    },
    'source_postprocess_result': postprocess_source.get('best_single_frame_metrics', {}),
}

EXPORT_META_PATH.write_text(json.dumps(EXPORT_META, ensure_ascii=False, indent=2), encoding='utf-8')
POSTPROCESS_CONFIG_PATH.write_text(json.dumps(DRIVING_POSTPROCESS_CONFIG, ensure_ascii=False, indent=2), encoding='utf-8')

print('saved ONNX:', ONNX_PATH)
print('saved meta:', EXPORT_META_PATH)
print('saved postprocess config:', POSTPROCESS_CONFIG_PATH)

saved ONNX: ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\06_lane_model_finetuning\model_exports\finetune_v1\maplane_finetune_v1_fp32_raw.onnx
saved meta: ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\06_lane_model_finetuning\model_exports\finetune_v1\maplane_finetune_v1_export_meta.json
saved postprocess config: ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\06_lane_model_finetuning\model_exports\finetune_v1\maplane_finetune_v1_driving_postprocess_config.json


## 9. 실행 결과 메모

실행 정상 종료됨. 에러 없음.

생성된 파일:

```text
model_exports/finetune_v1/
  maplane_finetune_v1_fp32_raw.onnx
  maplane_finetune_v1_export_meta.json
  maplane_finetune_v1_driving_postprocess_config.json
```

핵심 결과:

```text
checkpoint: ckpt/best.pth
ONNX size: 43.95 MB
ONNX input: image [1, 3, 320, 800] tensor(float)
ONNX output: raw_predictions [1, 192, 78] tensor(float)
max_abs_diff: 1.907e-06
mean_abs_diff: 6.826e-08
p95_abs_diff: 1.937e-07
```

해석:

- `missing keys = 0`, `unexpected keys = 0`이므로 fine-tuned checkpoint가 config 구조에 정상 로드됨.
- PyTorch raw output과 ONNX Runtime raw output 차이가 매우 작으므로 FP32 ONNX 변환은 성공으로 판단함.
- ONNX 파일이 `best.pth`보다 작은 것은 정상임. checkpoint에는 optimizer/학습 메타/중복 state 등이 들어갈 수 있고, ONNX는 추론 그래프와 weight 중심으로 저장됨.
- export 시간이 짧은 것도 raw forward export라 이상하지 않음.
- 이 ONNX는 steering을 직접 내지 않고 `[1, 192, 78]` raw prediction을 냄. lane decode와 05에서 정한 driving postprocess는 Pi Python 코드에서 수행해야 함.

다음 작업:

1. 07 노트북에서 ONNX를 대상으로 실제 holdout 이미지 parity/latency를 확인함.
2. 이후 Pi에 ONNX + postprocess config를 올려 preview/dry-run 주행 스크립트로 연결함.
3. 필요하면 FP32 ONNX 이후 PTQ를 적용하여 Pi latency를 줄임.